# TD — pandas (bases)

## Objectif

L'analyse de journaux réseau se fait avec `pandas`, la bibliothèque standard
de manipulation de données en Python. Avant les TP « détection de beacon » et
« voyage impossible », ce TD installe les **bases** : charger un CSV, inspecter
un DataFrame, sélectionner, filtrer, transformer, joindre et agréger.

**Consignes :** complétez chaque fonction ou variable indiquée dans un encadré
« ✏️ Exercice », puis exécutez la cellule de tests qui suit. Les tests affichent
`… OK` quand tout passe.

## Préambule

In [ ]:
import pandas as pd


## 1. Le journal de connexions

### 1.1 Charger les données (`read_csv`)

Pour charger des données, on appelle `pd.read_csv`. Le paramètre `parse_dates` convertit
la colonne `timestamp` en vraies dates, ce qui permet ensuite d'extraire
l'heure ou de calculer des écarts.

- Doc : https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html

La cellule suivante charge le journal `connexions.csv` dans le DataFrame `connexions`.

In [ ]:
connexions = pd.read_csv("connexions.csv", parse_dates=["timestamp"])

La cellule suivante affiche un aperçu de ce qu'on vient de charger : les
dimensions (`.shape`), le type de chaque colonne (`.info()`) et les premières
lignes (`.head()`).

In [ ]:
print("shape :", connexions.shape)
print("\n-------\n")
connexions.info()
print("\n-------\n")
connexions.head()


### 1.2 Sélectionner une colonne

Une colonne se sélectionne avec `df["nom"]` : on obtient une **Series**, la
colonne du DataFrame (l'équivalent d'une liste indexée). C'est l'opération de
base pour isoler un champ à analyser.

- `df["col"]` : la colonne `col` (une `Series`).

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 1</legend>

Compléter la variable `ips` : elle contient la colonne `ip` de `connexions`.

</fieldset>

In [ ]:
# BEGIN
## Rappel : connexions : DataFrame du journal de connexions
# ips = ...
ips = connexions["ip"]
# END

In [ ]:
assert isinstance(ips, pd.Series), "le résultat doit être une Series"
assert ips.nunique() == 55, f"attendu 55 IP distinctes, obtenu {ips.nunique()}"
print("ex 1 OK :", ips.nunique(), "IP distinctes")

### 1.3 Sélectionner plusieurs colonnes

Pour isoler un sous-ensemble de colonnes, on passe une **liste** de noms entre
crochets : on obtient un **DataFrame** (et non une Series), limité aux colonnes
demandées.

- `df[["col1", "col2"]]` : un sous-DataFrame ne contenant que `col1` et `col2`.

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 2</legend>

Compléter la variable `identite` : elle contient les colonnes `username` et `ip` de `connexions`.

</fieldset>

In [ ]:
# BEGIN
## Rappel : connexions : DataFrame du journal de connexions
# identite = ...
identite = connexions[["username", "ip"]]
# END

In [ ]:
assert isinstance(identite, pd.DataFrame), "le résultat doit être un DataFrame (pas une Series)"
assert list(identite.columns) == ["username", "ip"], f"colonnes inattendues : {list(identite.columns)}"
print("ex 2 OK :", identite.shape[0], "lignes,", list(identite.columns))

### 1.4 Filtrer les lignes

Pour ne garder que certaines lignes, on construit un **masque booléen** : une
Series de `True`/`False` (une condition sur une colonne), qu'on passe entre
crochets. Seules les lignes où le masque vaut `True` sont conservées.

- `df[df["col"] == val]` : les lignes du dataframe dont les valeurs de la colonne `col` sont égales à `val`
- Combiner avec `&` (et), `|` (ou), `~` (non) — les parenthèses sont
  obligatoires.

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 3</legend>

Compléter la variable `conn_alice` : elle contient les connexions de `connexions` rattachées à `alice.dupont`.

</fieldset>

In [ ]:
# BEGIN
## Rappel : connexions : DataFrame du journal de connexions
# conn_alice = ...
conn_alice = connexions[connexions["username"] == "alice.dupont"]
# END

In [ ]:
assert len(conn_alice) == 15, f"attendu 15 connexions, obtenu {len(conn_alice)}"
assert (conn_alice["username"] == "alice.dupont").all()
print("ex 3 OK :", len(conn_alice), "connexions pour alice.dupont")

### 1.5 Assigner une nouvelle colonne

On ajoute une colonne en écrivant `df["nouvelle"] = ...`. La valeur peut être
constante ou calculée à partir des colonnes existantes. Ici, on extrait l'heure
de `timestamp` grâce à l'accesseur `.dt`, propre aux colonnes de dates.

- `df["nouvelle"] = ...` : ajoute (ou remplace) une colonne.
- `df["timestamp"].dt.hour` : l'heure (0-23) de chaque date.
- `.dt.date`, `.dt.minute`, `.dt.dayofweek`… : les autres composants.

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 4</legend>

Compléter la fonction `ajouter_heure()`.

</fieldset>

In [ ]:
def ajouter_heure(df: pd.DataFrame) -> pd.DataFrame:
    """Renvoie une copie de `df` avec une colonne `heure` (entier 0-23)."""
    # BEGIN
    # raise NotImplementedError
    out = df.copy()
    out["heure"] = out["timestamp"].dt.hour
    return out
    # END

In [ ]:
avec_heures = ajouter_heure(connexions)
assert "heure" in avec_heures.columns, "colonne `heure` absente"
assert set(avec_heures["heure"].unique()) == {8, 9, 13, 18, 19, 20}
print("ex 4 OK : heures présentes =", sorted(avec_heures["heure"].unique()))

## 2. Enrichir le journal

### 2.1 Transformer avec `.map`

Un journal ne contient souvent que des identifiants (ici, une IP). Pour les
traduire en information lisible, on utilise un **dictionnaire de correspondance**
puis `.map`, qui remplace chaque valeur par son image.

La cellule suivante charge une petite base de géolocalisation `ip_geo.csv` (IP → pays,
ville, coordonnées) dans le DataFrame `geo`.

In [ ]:
geo = pd.read_csv("ip_geo.csv")

On affiche ici les premières lignes de `geo`.

In [ ]:
geo.head()

- `colonne.map(dico)` : remplace chaque valeur par `dico[valeur]` (ou `NaN` si
  la clé est absente).
- `dict(zip(geo["ip"], geo["country"]))` : fabrique le dictionnaire ip → pays.

Deux fonctions sont à compléter, en deux temps.

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 5a</legend>

Compléter la fonction `dico_pays()` : elle construit le dictionnaire `ip → pays` depuis `geo`.

</fieldset>

In [ ]:
def dico_pays(geo: pd.DataFrame) -> dict[str, str]:
    """Renvoie le dictionnaire ip → pays construit depuis `geo`."""
    # BEGIN
    # raise NotImplementedError
    return dict(zip(geo["ip"], geo["country"]))
    # END

In [ ]:
d = dico_pays(geo)
assert d["192.0.2.48"] == "France"
print("ex 5a OK :", len(d), "IP dans le dictionnaire")

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 5b</legend>

Compléter la fonction `pays_de()` : elle ajoute la colonne `pays` à `df` en s'appuyant sur `dico_pays()`.

</fieldset>

In [ ]:
def pays_de(df: pd.DataFrame, geo: pd.DataFrame) -> pd.DataFrame:
    """Renvoie une copie de `df` avec une colonne `pays` (IP mappée vers son pays)."""
    # BEGIN
    # raise NotImplementedError
    out = df.copy()
    out["pays"] = out["ip"].map(dico_pays(geo))
    return out
    # END

In [ ]:
avec_pays = pays_de(connexions, geo)
assert "pays" in avec_pays.columns
assert not avec_pays["pays"].isna().any(), "des IP n'ont pas de pays"
assert (avec_pays[avec_pays["ip"] == "192.0.2.48"]["pays"] == "France").all()
print("ex 5b OK :", len(avec_pays["pays"].unique()), "pays distincts")

### 2.2 Transformer avec `.apply`

`.map` remplace par une valeur issue d'un dictionnaire. Quand la transformation
est **calculée** (une fonction, une règle), on utilise `.apply`, qui applique
une fonction à chaque valeur de la colonne.

On classe chaque connexion selon son heure avec la fonction `categoriser_heure`
(fournie) :

In [ ]:
def categoriser_heure(h: int) -> str:
    """Classe une heure : "nuit" (<8), "travail" (8-18), "soir" (>18)."""
    if h < 8:
        return "nuit"
    if h <= 18:
        return "travail"
    return "soir"


- `colonne.apply(f)` : applique `f` à chaque valeur, renvoie une Series.
- Pour une fonction sur toute la ligne : `df.apply(f, axis=1)`.

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 6</legend>

Compléter la variable `cats` : elle contient la catégorie (`categoriser_heure`) de chaque connexion, selon sa colonne `heure`.

</fieldset>

In [ ]:
avec_heures = ajouter_heure(connexions)
# BEGIN
## Rappel : avec_heures : connexions avec la colonne `heure`
# cats = ...
cats = avec_heures["heure"].apply(categoriser_heure)
# END

In [ ]:
assert isinstance(cats, pd.Series)
assert set(cats.unique()) == {"travail", "soir"}, f"catégories : {set(cats.unique())}"
assert (cats[avec_heures["heure"] == 20] == "soir").all()
print("ex 6 OK : catégories =", sorted(cats.unique()))

### 2.3 Joindre deux tables (`merge`)

La géolocalisation vit dans une table séparée (`ip_geo.csv`). Pour l'attacher
au journal, on fait une **jointure** (`merge`) sur la colonne commune `ip` :
chaque ligne du journal reçoit la latitude/longitude/pays de son IP.

- `pd.merge(gauche, droite, on="ip", how="left")` : joint sur `ip`.
- `how="left"` : garde toutes les lignes de gauche, même sans correspondance
  (les colonnes ajoutées vaudraient alors `NaN`).

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 7</legend>

Compléter la variable `df_geo` : elle contient `connexions` jointe à `geo` sur `ip`.

</fieldset>

In [ ]:
# BEGIN
## Rappel : connexions, geo : DataFrames du journal et de la géolocalisation
# df_geo = ...
df_geo = connexions.merge(geo, on="ip", how="left")
# END

In [ ]:
assert "latitude" in df_geo.columns and "city" in df_geo.columns
assert df_geo.shape[1] == 7, f"attendu 7 colonnes, obtenu {df_geo.shape[1]}"
assert not df_geo["latitude"].isna().any(), "des IP sans géolocalisation"
print("ex 7 OK :", df_geo.shape[1], "colonnes,", df_geo.shape[0], "lignes")

## 3. Analyser les logs réseau

### 3.1 Grouper et agréger (`groupby` + `agg`)

Reste l'opération reine de l'analyse de logs : **regrouper** les lignes par une
clé puis **agréger** (compter, moyenner…) chaque groupe. C'est ainsi qu'on repère
les domaines les plus contactés ou les hôtes les plus bavards.

La cellule suivante charge `logs_reseau.csv`, le jeu de connexions sortantes du TP
« détection de beacon », dans le DataFrame `logs`.

In [ ]:
logs = pd.read_csv("logs_reseau.csv", parse_dates=["timestamp"])

On affiche ici les premières lignes de `logs`.

In [ ]:
logs.head()

- `df.groupby("clé")` : regroupe les lignes par valeur de `clé`.
- `.agg(n=("col", "count"), moyenne=("col", "mean"))` : agrège chaque groupe,
  une colonne de résultat par agrégation.

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 8</legend>

Compléter la fonction `trafic_par_domaine()`.

</fieldset>

In [ ]:
def trafic_par_domaine(df: pd.DataFrame) -> pd.DataFrame:
    """Par domaine : nombre de connexions et taille moyenne, triés par nombre décroissant."""
    # BEGIN
    # raise NotImplementedError
    stats = df.groupby("domain").agg(
        n=("timestamp", "count"),
        octets_moyen=("bytes_out", "mean"),
    )
    return stats.sort_values("n", ascending=False)
    # END

In [ ]:
stats = trafic_par_domaine(logs)
assert stats.index.name == "domain"
assert "n" in stats.columns and "octets_moyen" in stats.columns
assert stats["n"].iloc[0] == 396, f"attendu 396 en tête, obtenu {stats['n'].iloc[0]}"
assert stats.index[0] == "av-updates.example.net"
assert stats.loc["sync-updates.example.net", "octets_moyen"] == 512.0
print("ex 8 OK :", len(stats), "domaines ; en tête :", stats.index[0])

### 3.2 Combiner filtre, agrégation et sélection

En analyse de journaux, on enchaîne rarement une seule opération : on **filtre**
les lignes qui nous intéressent, on **agrège** les groupes obtenus, puis on
**sélectionne** les colonnes à conserver.

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 9</legend>

Compléter la fonction `contacts_domaine()`, qui enchaîne trois étapes :

1. filtrer les connexions vers `domaine` (colonne `domain`) ;
2. agréger par `src_ip` : nombre de connexions (`count`), total (`sum`) et
   moyenne (`mean`) des octets ;
3. ne garder que les colonnes `connexions` et `octets`, triées par octets
   décroissants.

</fieldset>

In [ ]:
def contacts_domaine(df: pd.DataFrame, domaine: str) -> pd.DataFrame:
    """Pour les connexions vers `domaine` : par poste source:
    - nombre de connexions,
    - total d'octets sortants
    - nombre moyen d'octets sortants
    triés par total d'octets décroissants.
    """
    # BEGIN
    # raise NotImplementedError
    filtre = df[df["domain"] == domaine]
    stats = filtre.groupby("src_ip").agg(
        connexions=("timestamp", "count"),
        octets=("bytes_out", "sum"),
        octets_moyen=("bytes_out", "mean"),
    )
    return stats.sort_values("octets", ascending=False)[["connexions", "octets"]]
    # END

In [ ]:
res = contacts_domaine(logs, "windowsupdate.com")
assert res.index.name == "src_ip"
assert list(res.columns) == ["connexions", "octets"], f"colonnes : {list(res.columns)}"
assert (res["connexions"] == 24).all()
assert res["octets"].sum() == 21953091
assert res["octets"].iloc[0] == 2796800
print("ex 9 OK :", len(res), "postes, total =", res["octets"].sum(), "octets")

## Bilan

| notion | fonction pandas | exercice |
| --- | --- | --- |
| charger un CSV | `pd.read_csv(..., parse_dates=...)` | 1.1 |
| inspecter | `.head()`, `.info()`, `.shape` | 1.1 |
| sélectionner une colonne | `df["col"]` | 1 |
| sélectionner plusieurs colonnes | `df[["col1", "col2"]]` | 2 |
| filtrer | `df[masque_booléen]` | 3 |
| assigner une colonne | `df["nouvelle"] = ...` | 4 |
| transformer par correspondance | `.map(dico)` | 5a, 5b |
| transformer par calcul | `.apply(f)` | 6 |
| joindre | `pd.merge(..., on=...)` | 7 |
| grouper + agréger | `.groupby(...).agg(...)` | 8 |
| combiner filtre + agrégation + sélection | enchaînement des trois | 9 |

**La suite.** Ces notions sont exactement celles des deux TP qui suivent :

- **détection de beacon** : `groupby` + `diff` + `agg` pour mesurer la régularité
  des connexions vers un C2 ;
- **voyage impossible** : `merge`, `map`, `isin` et `groupby().nunique()` pour
  détecter des déplacements impossibles et écarter les faux positifs.